In [ ]:
import pandas as pd
import regex as re
import os
import unicodedata

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df = pd.read_csv('stt_dataset/metadata.csv', delimiter="\t")
# df

In [ ]:
folder_path = 'stt_dataset/wavs'
file_count = sum(1 for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)))
print(f"Количество файлов в папке 'wavs': {file_count}")
print(f"Количество записей в df: {len(df)}")

In [ ]:
wavs_list = os.listdir(folder_path)

to_drop = set()
for idx, row in df.iterrows():
    if row['audio_file'] not in wavs_list:
        to_drop.add(idx)

lat_to_cyr = {
    "a": "а", "A": "А",
    "e": "е", "E": "Е",
    "i": "и", "I": "И",
    "o": "о", "O": "О",
    "u": "у", "U": "У",
    "y": "ы", "Y": "Ы",
}

punct_map = {
        '“': '"', '”': '"', '„': '"', '«': '"', '»': '"',
        '‘': "'", '’': "'", '‚': "'",
        '—': '-', '–': '-', '−': '-',
    }

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFD", str(text))  # 1. разложение
    
    for lat, cyr in lat_to_cyr.items():
        text = text.replace(lat, cyr)
    
    text = unicodedata.normalize("NFC", text)

    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C')

    text = re.sub(r'\s+', ' ', text)
    
    text = text.strip()
    
    for k, v in punct_map.items():
        text = text.replace(k, v)
    
    return text
    
df['text'] = df['text'].apply(normalize_text)

In [ ]:
df = df.drop(to_drop)

df = df.drop_duplicates(subset=['text'])
df = df.dropna()

print(f"Количество записей в df: {len(df)}")

In [ ]:
wav_to_drop = []

audio_list = df['audio_file'].tolist()
for wav in wavs_list:
    if wav not in audio_list:
        wav_to_drop.append(wav)

print(f"Количество файлов в папке 'wavs': {len(wavs_list) - len(wav_to_drop)}")

In [ ]:
folder_path = 'stt_dataset/wavs'

for wav in wav_to_drop:
    file_path = os.path.join(folder_path, wav)
    if os.path.isfile(file_path):
        os.remove(file_path)

print(f"{len(wav_to_drop)} файлов удалено")

df.to_csv('stt_dataset/metadata.csv', index=False, sep="\t")